# Comparing PCA, t-SNE, and UMAP

Companion notebook for Sections 5-9 of the lecture notes.

Objectives:

- run PCA, t-SNE, and optionally UMAP on the same dataset;
- compare runtime and local-neighborhood preservation;
- summarize when each method is appropriate;
- practice cautious interpretation of embeddings;
- map the lecture learning goals to runnable code.


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True


In [ ]:
import os
os.environ.setdefault('NUMBA_CACHE_DIR', '/tmp/numba-cache')

try:
    import umap.umap_ as umap
    UMAP_AVAILABLE = True
except ImportError:
    umap = None
    UMAP_AVAILABLE = False

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.preprocessing import StandardScaler

X_all, y_all = load_digits(return_X_y=True)
rng = np.random.default_rng(123)
idx = rng.choice(len(X_all), size=500, replace=False)
X = X_all[idx]
y = y_all[idx]
X_scaled = StandardScaler().fit_transform(X)

print('Dataset:', X_scaled.shape)
print('UMAP available:', UMAP_AVAILABLE)


## 1. Run the methods

PCA is linear and deterministic. t-SNE and UMAP are nonlinear embedding methods focused on local neighborhoods.


In [ ]:
embeddings = {}
summary_rows = []

def record(name, fit_func):
    t0 = time.perf_counter()
    emb = fit_func()
    elapsed = time.perf_counter() - t0
    embeddings[name] = emb
    summary_rows.append({
        'method': name,
        'runtime_seconds': elapsed,
        'trustworthiness_k10': trustworthiness(X_scaled, emb, n_neighbors=10),
    })

record('PCA', lambda: PCA(n_components=2, random_state=42).fit_transform(X_scaled))
record('t-SNE', lambda: TSNE(
    n_components=2,
    perplexity=30,
    init='pca',
    learning_rate='auto',
    max_iter=500,
    random_state=42,
).fit_transform(X_scaled))

if UMAP_AVAILABLE:
    record('UMAP', lambda: umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42).fit_transform(X_scaled))
else:
    print('UMAP skipped because umap-learn is not installed.')

pd.DataFrame(summary_rows).sort_values('trustworthiness_k10', ascending=False)


## 2. Visual comparison

Use the same data and colors. Compare local mixing/separation, but do not overinterpret axes or faraway cluster distances.


In [ ]:
fig, axes = plt.subplots(1, len(embeddings), figsize=(5 * len(embeddings), 4.5))
if len(embeddings) == 1:
    axes = [axes]

for ax, (name, emb) in zip(axes, embeddings.items()):
    scatter = ax.scatter(emb[:, 0], emb[:, 1], c=y, cmap='tab10', s=11, alpha=0.8)
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()


## 3. Practical comparison table


In [ ]:
comparison = pd.DataFrame([
    {
        'method': 'PCA',
        'linear': 'yes',
        'main_goal': 'preserve global variance/projection geometry',
        'transforms_new_data': 'yes',
        'typical_use': 'preprocessing, compression, denoising, simple visualization',
        'axis_meaning': 'sometimes interpretable as linear components',
        'trust_far_cluster_distances': 'only if a linear projection is appropriate',
    },
    {
        'method': 't-SNE',
        'linear': 'no',
        'main_goal': 'preserve local probabilistic neighborhoods',
        'transforms_new_data': 'usually no',
        'typical_use': 'exploratory visualization',
        'axis_meaning': 'no',
        'trust_far_cluster_distances': 'no',
    },
    {
        'method': 'UMAP',
        'linear': 'no',
        'main_goal': 'preserve local graph/manifold neighborhoods',
        'transforms_new_data': 'yes in many implementations',
        'typical_use': 'visualization, sometimes nonlinear preprocessing',
        'axis_meaning': 'no',
        'trust_far_cluster_distances': 'with care',
    },
])
comparison


## 4. Stability check

PCA is deterministic for a fixed dataset. t-SNE and UMAP use randomized optimization unless a seed is set. Even with the same seed, the coordinates themselves are less meaningful than the neighborhood pattern.


In [ ]:
stability_rows = []

pca_a = PCA(n_components=2).fit_transform(X_scaled)
pca_b = PCA(n_components=2).fit_transform(X_scaled)
stability_rows.append({'method': 'PCA', 'mean_abs_coordinate_difference': np.mean(np.abs(pca_a - pca_b))})

for state in [0, 1]:
    emb = TSNE(n_components=2, perplexity=30, init='random', learning_rate='auto', max_iter=500, random_state=state).fit_transform(X_scaled)
    stability_rows.append({'method': f't-SNE random_state={state}', 'mean_abs_coordinate_difference': np.nan})

if UMAP_AVAILABLE:
    for state in [0, 1]:
        emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=state).fit_transform(X_scaled)
        stability_rows.append({'method': f'UMAP random_state={state}', 'mean_abs_coordinate_difference': np.nan})

pd.DataFrame(stability_rows)


## 5. Method-selection rules

Use PCA when you need a simple, linear, reusable transformation. Use t-SNE when the goal is exploratory visualization of local neighborhoods. Use UMAP when you want a nonlinear neighborhood-preserving embedding and may need faster runtime or a reusable `transform` operation.


## Exercises

1. Change the dataset subset size from 500 to 1000. Which method slows down most?
2. Compare trustworthiness for `n_neighbors=5`, `10`, and `30`.
3. Pick one pair of classes that overlaps in PCA but separates in t-SNE or UMAP. What additional evidence would you need before claiming that the classes form real clusters?


## Takeaways

- PCA, t-SNE, and UMAP all reduce dimensionality but optimize different objectives.
- Runtime, transformability, and interpretability differ across methods.
- Nonlinear embeddings are useful visual tools, not literal maps of all original distances.
